# DeepSORT Extended — Google Colab

Запуск трекера на ваших данных с Google Drive.

**Структура данных на Drive:**
```
MyDrive/Videos-CV/
  MOT16-09/
    img/     ← картинки jpg (000001.jpg, 10.jpg, ...)
    gt/      ← txt с разметкой (gt.txt или любой .txt)
    det/     ← txt с детекциями (det.txt или любой .txt)
  Kitti-17/
  ...
```

**Runtime:** GPU (T4).

In [ ]:
# 1. Проверка GPU
!nvidia-smi

In [ ]:
# 2. Клонирование репозитория и установка зависимостей
REPO_URL = "https://github.com/danvlak/deep-sort-project.git"  # <-- замените на свой репозиторий

import os
if not os.path.exists("deep-sort-project"):
    !git clone $REPO_URL deep-sort-project
%cd deep-sort-project
!pip install -q -r requirements.txt

In [ ]:
# 3. Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Корневая папка с видео (последовательностями)
DATA_ROOT = "/content/drive/MyDrive/Videos-CV"

import os
assert os.path.isdir(DATA_ROOT), f"Папка не найдена: {DATA_ROOT}"
print("Доступные последовательности:")
for name in sorted(os.listdir(DATA_ROOT)):
    path = os.path.join(DATA_ROOT, name)
    if os.path.isdir(path):
        has_img = any(os.path.isdir(os.path.join(path, d)) for d in ("img", "img1", "images"))
        print(f"  - {name}" + ("  ✓" if has_img else "  (нет папки img)"))

In [ ]:
# 4. Настройки: выберите папку и модели
import os
from utils.mot_paths import find_sequence_dir, get_image_dir, get_gt_file, get_det_file, list_image_filenames

# Имя папки на Google Drive (как она называется у вас)
SEQUENCE_FOLDER = "MOT16-09"   # например: MOT16-09, Kitti-17, TUD-Campus

# Имя для сохранения результатов (стандартное имя из задания)
SEQUENCE = "MOT16-09"          # для KITTI-17 пишите "KITTI-17", даже если папка Kitti-17

DETECTOR = "yolov8n"           # yolov8n | yolov5s | rtdetr_r18
REID = "osnet_x0_25"           # osnet_x0_25 | resnet50_ibn | fastreid_sbs
CONFIG = "configs/default.yaml"

# Найти папку последовательности (поддерживает Kitti-17 и KITTI-17)
SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Папка последовательности:", SEQ_DIR)
print("Картинки:", get_image_dir(SEQ_DIR))
print("GT файл:", get_gt_file(SEQ_DIR), "(существует:", os.path.exists(get_gt_file(SEQ_DIR)), ")")
print("DET файл:", get_det_file(SEQ_DIR), "(существует:", os.path.exists(get_det_file(SEQ_DIR)), ")")
print("Количество кадров:", len(list_image_filenames(SEQ_DIR)))

### Что означают файлы gt и det?

- **gt** (ground truth) — правильная разметка: кто где находится на каждом кадре.
  Пример: `10,1,278,447,104,265,1,1,1` → кадр 10, человек ID=1, рамка (x,y,w,h).
- **det** (detections) — готовые детекции (рамки людей от детектора). Наш новый пайплайн **не использует** det-файлы — он сам детектирует людей через YOLO. Det нужен только для baseline (оригинальный DeepSORT).

In [ ]:
# 5. Запуск трекинга на одной последовательности
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config {CONFIG} \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

In [ ]:
# 6. Генерация overlay-видео
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video
overlay_path = f"results/overlays/{SEQUENCE}_overlay.mp4"
if os.path.exists(overlay_path):
    display(Video(overlay_path, embed=True))
else:
    print("Видео не найдено:", overlay_path)

In [ ]:
# 7. Полная оценка на всех 6 видео (долго, ~1-2 часа)
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    !python eval/run_benchmark.py \
      --mot_root "{DATA_ROOT}" \
      --output_dir results/modern \
      --detector {DETECTOR} \
      --reid {REID}

    !python eval/run_mot.py \
      --mot_dir "{DATA_ROOT}" \
      --output_dir results/modern \
      --skip_tracking

    !python eval/eval_detector.py \
      --mot_dir "{DATA_ROOT}" \
      --detector {DETECTOR}

    !python eval/eval_reid.py \
      --mot_dir "{DATA_ROOT}" \
      --reid {REID}

## Список видео для задания

| Папка на Drive (пример) | SEQUENCE для результатов |
|---|---|
| TUD-Campus | TUD-Campus |
| TUD-Stadtmitte | TUD-Stadtmitte |
| Kitti-17 | KITTI-17 |
| PETS09-S2L1 | PETS09-S2L1 |
| MOT16-09 | MOT16-09 |
| MOT16-11 | MOT16-11 |

Для каждого видео поменяйте `SEQUENCE_FOLDER` и `SEQUENCE` в ячейке 4 и перезапустите ячейки 4–6.